In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt 
import ast

## Data cleaning

Exploration of datasets to verify the accuracy of the data, to perform necessary conversions, to analyze duplicate information and NaN values and their management, with possible elimination of empty columns and replacement of NaNs with medians or relevant data to the analysis.

### Character anime works

In [2]:
character_anime_works = pd.read_csv(r'../data/character_anime_works.csv')
character_anime_works.tail()

,anime_mal_id,character_mal_id,character_name,role
236811,31245,137157,"Shibasaki, Ken",Supporting
236812,36305,136064,"Hamanaka, Midori",Main
236813,36305,133916,"Narumi, Sena",Main
236814,36305,124942,"Hayasaka, Akari",Supporting
236815,34895,155887,"Shiraha, Setsuna",Main


This dataset represents a bridge table between Anime and Characters, especially the "who plays what" relationship and defines the nature of that participation through the role column.

In [3]:
character_anime_works.shape

(236816, 4)

In [4]:
character_anime_works.columns

Index(['anime_mal_id', 'character_mal_id', 'character_name', 'role'], dtype='object')

In [5]:
character_anime_works.dtypes

anime_mal_id         int64
character_mal_id     int64
character_name      object
role                object
dtype: object

In [6]:
character_anime_works['role'].value_counts()

role
Supporting    186285
Main           50531
Name: count, dtype: int64

Observations:
- **Relational structure**: essential to let us answer to qestions about the highest number of main character of an anime and many more.
- **Role distribution**: natural assignment of supporting and main roles in a narrative dataset.

To do:
- **Memory usage**: if anime_mal_id and character_mal_id have values representable using int32 we can save memory space. Furthermore, 'role' column has two standard values, it'd be more efficient if we could convert it into 'category' type.
- **Duplicates management**: an essential requirement includes if the same character is associated to the same anime more than once. This could be a repeated information error.
- **NaN analysis**: verify potentially ID missing in some rows to avoid existence of dirty data.

In [7]:
# Memory usage
# Checking max values (Scaling up strategy)
print(f"Max anime_id: {character_anime_works['anime_mal_id'].max()}")
print(f"Max character_id: {character_anime_works['character_mal_id'].max()}")

# Saving memory space
character_anime_works['anime_mal_id'] = character_anime_works['anime_mal_id'].astype('int32')
character_anime_works['character_mal_id'] = character_anime_works['character_mal_id'].astype('int32')

Max anime_id: 62584
Max character_id: 282284


In [8]:
character_anime_works.dtypes

anime_mal_id         int32
character_mal_id     int32
character_name      object
role                object
dtype: object

In [9]:
# Type conversion for higher efficiency
character_anime_works['role'] = character_anime_works['role'].astype('category')

# Memory usage
character_anime_works.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 236816 entries, 0 to 236815
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   anime_mal_id      236816 non-null  int32   
 1   character_mal_id  236816 non-null  int32   
 2   character_name    236816 non-null  object  
 3   role              236816 non-null  category
dtypes: category(1), int32(2), object(1)
memory usage: 3.8+ MB


In [10]:
# Duplicates management - key couple
duplicates = character_anime_works.duplicated(subset=['anime_mal_id', 'character_mal_id']).sum()
print(f"Relazioni duplicate: {duplicates}")

# Removal keeping the first occurency
character_anime_works = character_anime_works.loc[~character_anime_works.duplicated(subset=['anime_mal_id', 'character_mal_id'])]

Relazioni duplicate: 0


In [11]:
# NaN analysis
# Counting NaNs in key column
critical_nans = character_anime_works[['anime_mal_id', 'character_mal_id']].isna().sum()
print("NaN in colonne critiche:\n", critical_nans)

# Removing rows due to unimportant meaning
# character_anime_works = character_anime_works.dropna(subset=['anime_mal_id', 'character_mal_id'])

NaN in colonne critiche:
 anime_mal_id        0
character_mal_id    0
dtype: int64


Results:
- **Lighter**: Memory requirements will be reduced by approximately 60-70% thanks to the int32 and category types.

- **More robust**: We'll be sure that every line represents a valid relationship between an anime and a character.

- **User-centered**: We've removed "noise" (duplicates and potential null IDs) to allow journalists to draw accurate conclusions.

In [12]:
# Overwriting/importing clean dataframe into .csv
character_anime_works.to_csv('../data/character_anime_works_clean.csv', index=False)
print(f"\nSaved clean characher_anime_works dataset")


Saved clean characher_anime_works dataset


In [48]:
character_anime_works = None
print(f"\nRemoving charachers_anime_works dataset from RAM")


Removing charachers_anime_works dataset from RAM


Implementation ideas:
1. Counting how much anime each character act in
2. Which anime has the most crowded cast; a journalist may desire to know if anime with more characters in has higher or lower ratings cross-checking data with details.csv
3. Creating a filtered view with only main characters

### Characters' details

In [23]:
characters = pd.read_csv(r'../data/characters.csv')
characters

,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN
...,...,...,...,...,...,...,...
209958,282276.0,https://myanimelist.net/character/282276/Farra...,Farrah Van Dorothy,ファラ・ヴァン・ドロシー,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209959,282277.0,https://myanimelist.net/character/282277/Harri...,Harris Mead,ハリス・ミード,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209960,282278.0,https://myanimelist.net/character/282278/Rob,Rob,ロブ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209961,282281.0,https://myanimelist.net/character/282281/Grimm,Grimm,グリム,https://cdn.myanimelist.net/images/characters/...,0.0,NaN


In [24]:
characters.shape

(209963, 7)

In [25]:
characters.columns

Index(['character_mal_id', 'url', 'name', 'name_kanji', 'image', 'favorites',
       'about'],
      dtype='object')

In [26]:
characters.dtypes

character_mal_id    float64
url                  object
name                 object
name_kanji           object
image                object
favorites           float64
about                object
dtype: object

Duplicated ID - repeated informations error.

Based on our notes during the course, in this case we should remove them keeping only the first occurrence to avoid changing mean and other statistics

In [27]:
# Counts duplicated ID
duplicate_count = characters['character_mal_id'].duplicated().sum()
print(f"ID duplicati trovati: {duplicate_count}")

# To see duplicated rows
if duplicate_count > 0:
    print(characters[characters['character_mal_id'].duplicated(keep=False)].sort_values(by='character_mal_id'))

ID duplicati trovati: 456
        character_mal_id                                                url  \
208187             546.0       https://myanimelist.net/character/546/Hayate   
208185             546.0       https://myanimelist.net/character/546/Hayate   
208186             546.0       https://myanimelist.net/character/546/Hayate   
204358            4527.0  https://myanimelist.net/character/4527/Narue_N...   
204356            4527.0  https://myanimelist.net/character/4527/Narue_N...   
...                  ...                                                ...   
209850          282115.0    https://myanimelist.net/character/282115/Cavitt   
209859          282131.0  https://myanimelist.net/character/282131/Lieut...   
209858          282131.0  https://myanimelist.net/character/282131/Lieut...   
209733               NaN                                                NaN   
209734               NaN                                                NaN   

                     name

In [28]:
# Removing duplicates and keeping first record found
characters = characters.loc[~characters.duplicated(subset=['character_mal_id'])]

# Confirm current status
print(f"Dataset dopo la rimozione duplicati: {len(characters)} righe")

Dataset dopo la rimozione duplicati: 209507 righe


Column 'favorites' - meaning: describing how many times a user add the character to his/her/its favourites

In [29]:
characters['favorites'].describe()

count    209506.000000
mean         57.697216
std        1198.841827
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max      175632.000000
Name: favorites, dtype: float64

1. **High percentage of zero values**: usual event of large-scale dataset: a  few characters (the famous protagonists) have thousands of likes, while the vast majority of secondary characters have zero or very few of them.
2. **Type anomaly**: The dtype is float64, but the "likes" are necessary integers.

Solutions
- **Memory optimization**: convert column values into int32. Float64 value type takes up a lot of space.

- **Outlier management**: the maximum value of 175.632 is an important signal (the most loved character), not an error. Don't remove but use it to generate a categorization (es. "Top Tier", "Mainstream", "Niche") to encourage analysis for expert users.

- **Cardinality Filter**: If you're only interested in relevant people for your "Data Story", you could create a view of the data frame that excludes those with zero favorites, reducing the cardinality; this allows you to "work with seemingly smaller data" and focus only on the portion of data actually needed for popularity analysis.

In [36]:
# Memory optimization - replacing NaN values with 0 (not in favorites) and convert
characters.loc[:, 'favorites'] = characters['favorites'].fillna(0).astype('int32')
print(characters['favorites'].dtypes)

float64


In [37]:
# Outlier management - identify outlier categories
bins = [0, 1000, 10000, float('inf')]
label = ['Niche', 'Mainstream', 'Top Tier']
characters.loc[:, 'popularity'] = pd.cut(characters['favorites'], bins=bins, labels=label)
print(characters['popularity'].value_counts())

popularity
Niche         76268
Mainstream     1326
Top Tier        231
Name: count, dtype: int64


In [38]:
# Cardinality Filter - generating a view which avoid rows with 'favorites' values = 0
characters_fav_view = characters[characters['favorites'] > 0]

# If we don't need original data, reallocate to save memory
# characters = characters[characters['favorites'] > 0]

print(f"View rows (character with fav > 0): {len(characters_fav_view)}")

View rows (character with fav > 0): 77825


This convert the highest value to essential knowledge to the audience.

Column 'about' meaning - character's biography

In [39]:
characters['about'].value_counts()

about
No voice actors have been added to this character. Help improve our database by searching for a voice actor, and adding this character to their roles .                                                                                                                                                                                                                                                                                                                                                                                                                            199
Appears in episode 1.                                                                                                                                                                                                                                                                                                                                                                                                                      

**Data Interpretation**: the 199 occurrences of "No voice actors have been added..." and short descriptions like "Appears in episode 1" are signs of poor or missing data, even though they are not formally NaNs.

Thoughts
1. **Replacement**: Is it better to replace Nan with "No info"?
2. **Length Analysis**: Instead of reading the text, create a new column about_length. There's often a correlation between a character's popularity and the length of their description.
3. **Radical Memory Savings**: If you don't plan on doing text analysis, this column takes up a lot of RAM. Out notes suggest that if it's not needed for the current analysis, it's best to drop it or not load it at all to save space.

In [40]:
# Replacement - improves readability
characters.loc[:, 'about'] = characters['about'].fillna('No info')

If a journalist reads a profile, he/she'll be albe to understand the information more clearly than an empty space or a technical error.

In [42]:
# Total for each column
print(characters.isna().sum())

# Total of each row which contains at least a NaN value
print(f"Totale righe con almeno un NaN: {characters.isna().any(axis=1).sum()}")

character_mal_id         1
url                      1
name                     1
name_kanji           55369
image                    1
favorites                0
about                    0
popularity          131682
dtype: int64
Totale righe con almeno un NaN: 143990


The data extracted shows that 125,436 of 209,963 rows (about 60% of the dataset) have at least one null value.

What does this mean?

- **Massive Loss**: If you used a simple dropna(), you would delete more than half of your database.

- **Nature of the data**: Most of the NaNs are found in _about_ (96,976) and _name_kanji_ (55,480). It's common for many characters to lack a biography or Japanese name in the database.

To do:
- **Avoid Bias**: If we remove a row due to a NaN in an irrelevant column (such as name_kanji if you're doing a statistical analysis of favorites), out analysis will be highly biased, i.e., incorrect.

- **Leverage Pandas' design**: Pandas is designed to skip NaNs during numeric operations, so their presence will not affect the calculation of the favorites average.

- **Critical columns**: Only remove rows if the NaN is present in character_mal_id or name, since without this data the element becomes meaningless.

In [43]:
# We only remove if essential identity data is missing
characters = characters.dropna(subset=['character_mal_id'])

# For the others, we do not delete anything to avoid bias
print(f"Percentuale di dati preservati: {len(characters)/209963*100:.2f}%")

Percentuale di dati preservati: 99.78%


In [44]:
# Overwriting/importing clean dataframe into .csv
characters.to_csv('../data/characters_clean.csv', index=False)
print(f"\nSaved clean charachers dataset")


Saved clean charachers dataset


In [47]:
characters = None
print(f"\nRemoving charachers dataset from RAM")


Removing charachers dataset from RAM


### Details

In [14]:
details_df = pd.read_csv(r'../data/Details.csv')
details_df.tail()

,mal_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,...,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming
28950,59421,Zutaboro Reijou wa Ane no Moto Konyakusha ni D...,ずたぼろ令嬢は姉の元婚約者に溺愛される,https://myanimelist.net/anime/59421/Zutaboro_R...,https://cdn.myanimelist.net/images/anime/1518/...,TV,Finished Airing,7.37,15624.0,2025-07-05T00:00:00+00:00,...,['Josei'],Light novel,PG-13 - Teens 13 or older,12.0,summer,2025.0,"['Studio Pierrot', 'Mainichi Broadcasting Syst...",[],[],"['Crunchyroll', 'Aniplus TV', 'Bahamut Anime C..."
28951,31245,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,ずっと前から好きでした。～告白実行委員会～,https://myanimelist.net/anime/31245/Zutto_Mae_...,https://cdn.myanimelist.net/images/anime/3/821...,Movie,Finished Airing,7.20,104106.0,2016-04-23T00:00:00+00:00,...,[],Music,PG-13 - Teens 13 or older,1.0,NaN,NaN,"['Aniplex', 'Dentsu', 'Kadokawa Shoten', 'Movi...",[],['Aniplex of America'],[]
28952,36305,Zutto Mae kara Suki deshita. Kokuhaku Jikkou I...,ずっと前から好きでした。～告白実行委員会～ 「金曜日のおはよう」,https://myanimelist.net/anime/36305/Zutto_Mae_...,https://cdn.myanimelist.net/images/anime/6/883...,Special,Finished Airing,7.17,10038.0,2016-10-26T00:00:00+00:00,...,[],Music,PG - Children,1.0,NaN,NaN,['Aniplex'],[],[],[]
28953,34895,Zutto Suki Datta,ずっと好きだった,https://myanimelist.net/anime/34895/Zutto_Suki...,https://cdn.myanimelist.net/images/anime/1498/...,OVA,Finished Airing,5.68,1887.0,2017-04-21T00:00:00+00:00,...,[],Manga,Rx - Hentai,2.0,NaN,NaN,"['Queen Bee', 'Mediabank']",[],[],[]
28954,56606,Zutto Tomodachi,ずっと友達,https://myanimelist.net/anime/56606/Zutto_Tomo...,https://cdn.myanimelist.net/images/anime/1462/...,Music,Finished Airing,NaN,NaN,1996-06-01T00:00:00+00:00,...,['Kids'],Original,G - All Ages,1.0,NaN,NaN,['NHK'],[],[],[]


### GESTIONE ANNO DI RILASCIO, DUPLICATI E CONVERSIONE DATE
In questa fase preliminare verifichiamo l'unicità dei dati attraverso l'identificativo univoco mal_id. Sono presenti 2 coppie di duplicati attraverso gli'identificativi 'title' e 'type', andando a verificare hanno 'mal_id' e 'start-date' differenti quindi non li consideriamo duplicati.
Successivamente, convertiamo le colonne temporali (start_date, end_date) nel formato corretto datetime. Sovrascriviamo la colonna originale year, utilizzando l'anno da start_date, poichè presentava un alto numero di valori mancanti. Utilizziamo errors='coerce' per trasformare eventuali date anomale in NaT (Not a Time), garantendo la coerenza del dataset.


In [15]:
initial_rows = len(details_df)
details_df.drop_duplicates(subset='mal_id', keep='first', inplace=True)
print(f"Duplicati rimossi: {initial_rows - len(details_df)}")

details_df['start_date'] = pd.to_datetime(details_df['start_date'], errors='coerce')
details_df['end_date'] = pd.to_datetime(details_df['end_date'], errors='coerce')
details_df['year'] = details_df['start_date'].dt.year
details_df['year'] = details_df['year'].astype('Int64')

subset_check = ['title', 'type']
duplicates_mask = details_df.duplicated(subset=subset_check, keep=False)

num_duplicates = duplicates_mask.sum()
print(f"Possibili duplicati trovati (stesso Titolo e Tipo): {num_duplicates}")
if num_duplicates > 0:
    print("\nEsempio di righe duplicate (Titolo + Tipo):")
    # Ordiniamo per titolo così vediamo le coppie vicine
    display(details_df[duplicates_mask].sort_values(by='title')[['mal_id', 'title', 'type', 'start_date']].head(10))

Duplicati rimossi: 0
Possibili duplicati trovati (stesso Titolo e Tipo): 4

Esempio di righe duplicate (Titolo + Tipo):


,mal_id,title,type,start_date
16205,59994,Moonlight,Music,2024-09-30 00:00:00+00:00
16206,60524,Moonlight,Music,2024-12-09 00:00:00+00:00
21601,53367,Shen Lan Qi Yu Wushuang Zhu: Tianmo Pian,ONA,2023-01-14 00:00:00+00:00
21602,55740,Shen Lan Qi Yu Wushuang Zhu: Tianmo Pian,ONA,2023-07-15 00:00:00+00:00


In [16]:
print(details_df.dtypes)

mal_id                           int64
title                           object
title_japanese                  object
url                             object
image_url                       object
type                            object
status                          object
score                          float64
scored_by                      float64
start_date         datetime64[ns, UTC]
end_date           datetime64[ns, UTC]
synopsis                        object
rank                           float64
popularity                       int64
members                          int64
favorites                        int64
genres                          object
studios                         object
themes                          object
demographics                    object
source                          object
rating                          object
episodes                       float64
season                          object
year                             Int64
producers                

### CONVERSIONE STRINGHE E GESIONE VALORI MANCANTI
- In questa fase abbiamo trasformato le colonne come 'genres' e 'studios' utilizzando la libreria ast, convertendole in vere e proprie liste Python, permettendo al software di riconoscere ogni singolo genere o studio separatamente. Questa operazione è fondamentale per contare correttamente quante volte compare un genere o confrontare i vari studi di produzione tra loro.
- Nelle colonne convertite in liste, come i generi, i valori nulli sono stati sostituiti con liste vuote per mantenere la coerenza del codice e permettere calcoli sicuri. Per quanto riguarda le valutazioni numeriche come lo score, abbiamo scelto di conservare i valori mancanti invece di sostituirli con degli zeri, evitando così di abbassare artificialmente la media globale dei voti e preservando l'integrità della distribuzione.

In [17]:
# Funzione di conversione sicura
def to_list(x):
    if isinstance(x, str) and x.startswith('['):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return x if isinstance(x, list) else []

details_df['genres'] = details_df['genres'].apply(to_list)
details_df['studios'] = details_df['studios'].apply(to_list)
details_df['producers'] = details_df['producers'].apply(to_list)
details_df['licensors'] = details_df['licensors'].apply(to_list)
details_df['themes'] = details_df['themes'].apply(to_list)
details_df['demographics'] = details_df['demographics'].apply(to_list)

# Verifica del tipo di dato per una colonna
print(f"Tipo di dato in 'genres': {type(details_df['genres'].iloc[0])}")

Tipo di dato in 'genres': <class 'list'>


### RIMOZIONE RIGHE E COLONNE INUTILI
In quest'ultima fase abbiamo rimosso i record privi di informazioni essenziali come il titolo e il tipo di produzione, poiché la loro assenza impedirebbe qualsiasi classificazione corretta dell'opera. Abbiamo inoltre eliminato le colonne superflue (title_japanese, image_url, url, season) per ridurre l'occupazione di memoria e focalizzare l'analisi sulle variabili chiave. La verifica finale mostra un dataset solido di 28.888 righe, dove le lacune residue (come negli score) sono fisiologiche e verranno gestite puntualmente durante le analisi statistiche per non inquinare i risultati.

In [18]:
details_df.dropna(subset=['title', 'type'], inplace=True)

colonne_da_escludere = ['title_japanese','image_url', 'url', 'season']
details_df.drop(columns=colonne_da_escludere, axis=1, errors='ignore', inplace=True)

# 5. VERIFICA FINALE
null_counts = details_df.isna().sum()
print("Dataset pulito con successo!")
print(f"Righe rimaste: {len(details_df)}")
print("\nColonne con ancora valori mancanti (%):")
print((null_counts[null_counts > 0] / len(details_df)) * 100)

Dataset pulito con successo!
Righe rimaste: 28888

Colonne con ancora valori mancanti (%):
score         34.640681
scored_by     34.640681
start_date     2.748546
end_date      61.001108
synopsis      17.723622
rank          23.861119
rating         1.896981
episodes       2.132373
year           2.748546
dtype: float64


CONVERSIONE DA Int64 a Int32

**ID e Numerici**: 
Colonne come mal_id (l'identificativo unico di MyAnimeList) verranno convertite dal formato standard int64 al formato Int32.

**La scelta di Int32**: Utilizzerò il tipo "Nullable Integer" di Pandas (Int32 con la "I" maiuscola) invece del classico int32 di NumPy. Questa distinzione è vitale: il formato con la maiuscola permette di gestire eventuali valori mancanti (NaN) senza generare errori e senza forzare la colonna a diventare un pesante float64, risparmiando circa il 50% dello spazio in memoria per ogni cella.

**Categorizzazione**: La colonna type (che indica se l'anime è un TV, Movie, OVA, ecc. ) verrà convertita nel tipo category. Poiché i valori sono ripetitivi e limitati, questo approccio è molto più efficiente rispetto alla memorizzazione come stringhe generiche (object)

In [19]:
column_int64 = details_df.select_dtypes(include=['int64']).columns

details_df[column_int64] = details_df[column_int64].astype('Int32')

print("New type of data:")
print(details_df.dtypes)

New type of data:
mal_id                           Int32
title                           object
type                            object
status                          object
score                          float64
scored_by                      float64
start_date         datetime64[ns, UTC]
end_date           datetime64[ns, UTC]
synopsis                        object
rank                           float64
popularity                       Int32
members                          Int32
favorites                        Int32
genres                          object
studios                         object
themes                          object
demographics                    object
source                          object
rating                          object
episodes                       float64
year                             Int32
producers                       object
explicit_genres                 object
licensors                       object
streaming                       object
dtype: 

In [20]:
details_df.to_csv('../data/details_clean.csv', index=False)
print(f"\nSaved clean details dataset")



Saved clean details dataset


In [21]:
Details = None
print(f"\nRemoving dataset from RAM")


Removing dataset from RAM


### Users's favourite type

In [51]:
favs = pd.read_csv(r'../data/favs.csv')
favs

,username,fav_type,id
0,ishikawas,anime,45649
1,ishikawas,anime,38680
2,ishikawas,anime,795
3,ishikawas,anime,37510
4,ishikawas,anime,820
...,...,...,...
4178742,vincent0607,character,497
4178743,vincent0607,character,118739
4178744,vincent0607,character,188177
4178745,vincent0607,character,141354


In [52]:
favs.shape

(4178747, 3)

In [53]:
favs.columns

Index(['username', 'fav_type', 'id'], dtype='object')

In [54]:
favs.dtypes

username    object
fav_type    object
id           int64
dtype: object

It's an essential file to bridge user profiles (username) to what they love (id), filtered by type

1. **Understanding** favorites categories to eventually convert the data into category to save memory

In [55]:
favs['fav_type'].value_counts()

fav_type
character    1598040
anime        1531857
people        862186
company       186664
Name: count, dtype: int64

If fav_type contains only a few repeated strings, converting it to category will save huge amounts of RAM.

2. **Memory usage**: since the file has 4.1 million lines, optimization is a must to keep the system running smoothly for experienced journalists. We decided to go for an ID and type conversions

In [56]:
# Memory usage - conversion operations
favs['id'] = favs['id'].astype('int32')
favs['fav_type'] = favs['fav_type'].astype('category')

3. **Duplicates management**: in preference relationships, a user shouldn't be able to add the same item to their favorites more than once. This would be a "repetition error."

In [57]:
# Checking duplicates into username-type-id values
duplicates = favs.duplicated(subset=['username', 'fav_type', 'id']).sum()
print(f"Founded favorites duplicates: {duplicates}")

# Removal keeping the first occurency
if duplicates > 0:
    favs = favs.loc[~favs.duplicated(subset=['username', 'fav_type', 'id'])]

Founded favorites duplicates: 0


4. **NaN management**: a "favorite" without a username or ID has no informative value and should be removed because it would distort the analysis.

In [58]:
# Counting NaNs
print("NaN for each column:\n", favs.isna().sum())

# Removal rows with missed username/id
favs = favs.dropna(subset=['username', 'id'])

NaN for each column:
 username    4
fav_type    0
id          0
dtype: int64


In [59]:
# Overwriting/importing clean dataframe into .csv
favs.to_csv('../data/favs_clean.csv', index=False)
print(f"\nSaved clean favs dataset")


Saved clean favs dataset


In [61]:
favs = None
print(f"\nRemoving favs dataset from RAM")


Removing favs dataset from RAM


Implementation ideas:
1. **"Power users" analysis**: using 'favs['username'].value_counts().describe()' or 'print(favs['username'].value_counts().head(10))' to see mean adding favorites distribution for each user; Are there users with thousands of favorites?
2. **Popularity by type**: what community like most; relation between "anime" favorites and "character" ones?
3. **Most recurring ID**: cross-checking data with characters.csv we could give a name to most loved characters
4. **Cardinality filter**: creating a view based on characters

### Person details

In [62]:
person_details = pd.read_csv(r'../data/person_details.csv')
person_details

,person_mal_id,url,website_url,image_url,name,given_name,family_name,birthday,favorites,relevant_location
0,1,https://myanimelist.net/people/1/Tomokazu_Seki,NaN,https://cdn.myanimelist.net/images/voiceactors...,Tomokazu Seki,智一,関,1972-09-08T00:00:00+00:00,6219,"Berlin, Germany"
1,2,https://myanimelist.net/people/2/Tomokazu_Sugita,https://agrs.co.jp/,https://cdn.myanimelist.net/images/voiceactors...,Tomokazu Sugita,智和,杉田,1980-10-11T00:00:00+00:00,47666,"Los Angeles, USA"
2,3,https://myanimelist.net/people/3/Satsuki_Yukino,NaN,https://cdn.myanimelist.net/images/voiceactors...,Satsuki Yukino,さつき,ゆきの,1970-05-25T00:00:00+00:00,1777,"Madrid, Spain"
3,4,https://myanimelist.net/people/4/Aya_Hirano,http://ayahirano.jp/,https://cdn.myanimelist.net/images/voiceactors...,Aya Hirano,綾,平野,1987-10-08T00:00:00+00:00,18374,"Paris, France"
4,5,https://myanimelist.net/people/5/Kenichi_Suzumura,https://intention-k.com,https://cdn.myanimelist.net/images/voiceactors...,Kenichi Suzumura,健一,鈴村,1974-09-12T00:00:00+00:00,5176,"Osaka, Japan"
...,...,...,...,...,...,...,...,...,...,...
76694,90011,https://myanimelist.net/people/90011/Nanako_Ki...,NaN,https://cdn.myanimelist.net/img/sp/icon/apple-...,Nanako Kishimoto,七子,岸本,NaN,0,"Mumbai, India"
76695,90012,https://myanimelist.net/people/90012/Pamon,NaN,https://cdn.myanimelist.net/img/sp/icon/apple-...,Pamon,NaN,파몬,NaN,0,"Tokyo, Japan"
76696,90013,https://myanimelist.net/people/90013/Tomoru_Emoto,NaN,https://cdn.myanimelist.net/img/sp/icon/apple-...,Tomoru Emoto,ともる,柄本,NaN,0,"Tokyo, Japan"
76697,90014,https://myanimelist.net/people/90014/Hirari,https://hirari.2-d.jp/,https://cdn.myanimelist.net/img/sp/icon/apple-...,Hirari,NaN,ひらり,NaN,0,"Paris, France"


In [63]:
person_details.shape

(76699, 10)

In [64]:
person_details.columns

Index(['person_mal_id', 'url', 'website_url', 'image_url', 'name',
       'given_name', 'family_name', 'birthday', 'favorites',
       'relevant_location'],
      dtype='object')

In [65]:
person_details.dtypes

person_mal_id         int64
url                  object
website_url          object
image_url            object
name                 object
given_name           object
family_name          object
birthday             object
favorites             int64
relevant_location    object
dtype: object

**Key elements for storytelling**:

- **relevant_location**: Allows us to analyze the geography of talent (e.g., how many voice actors work in Tokyo vs. abroad).

- **favorites**: As with characters, this indicates the popularity of professionals.

- **birthday**: Essential for studying the average age of professionals over time.

In [66]:
# Memory usage - conversion operations
person_details['person_mal_id'] = person_details['person_mal_id'].astype('int32')
person_details['favorites'] = person_details['favorites'].astype('int32')

In [67]:
# Birthday column conversion to datetime type
person_details['birthday'] = pd.to_datetime(person_details['birthday'], errors='coerce')

In [68]:
# Duplicates management - ID
print(f"Duplicati ID persone: {person_details.duplicated(subset=['person_mal_id']).sum()}")

Duplicati ID persone: 105


105 duplicates on a primary key like person_mal_id indicates a "repetition error". In a relational database, each ID should uniquely identify a person.

- **Risk of Bias**: In this case, removing duplicates does not introduce bias, but rather removes distortion. If we retained duplicates, any statistics would be distorted because we would be counting the same person multiple times.

In [69]:
# Removal duplicates based on ID
initial_len = len(person_details)
person_details = person_details.loc[~person_details.duplicated(subset=['person_mal_id'])]

# Visualizing result
removed = initial_len - len(person_details)
print(f"Removed rows due to ID duplicates: {removed}")

Removed rows due to ID duplicates: 105


In [70]:
# NaN counting values for each column
print("Conteggio NaN:\n", person_details.isna().sum())

Conteggio NaN:
 person_mal_id            0
url                      0
website_url          59186
image_url                0
name                     2
given_name           30211
family_name          18610
birthday             59399
favorites                0
relevant_location        0
dtype: int64


website_url (59,264 NaN) and birthday (59,473 NaN): Approximately 77-78% of the dataset is missing this information.

- **Inference**: Many voice actors or other professionals don't have an official website or their birthdates aren't public.

- **Beware of Bias**: We absolutely must not remove these rows. If we did, we would drastically reduce the cardinality of the dataset (from 76k to less than 17k), losing valuable data on thousands of professionals simply because we don't know their birthdays.

- **given_name and family_name**: Many professionals use stage names or a single name. Their absence isn't a critical error.

1. **Check for at least one name**: to ensure that a record has at least a partial identity.

In [71]:
# Finding rows where both given_name and family_name are missing
both_names_missing = person_details['given_name'].isna() & person_details['family_name'].isna() & person_details['name'].isna()

# Visualizing them
print(f"Professionisti senza alcun nome (given e family e name): {both_names_missing.sum()}")

Professionisti senza alcun nome (given e family e name): 1


Due to this research, we can conclude none of the name fields is full in only one row, so we can use imputation rule and build the name in the other row where there isn't a name but one of another two name types.

If a professional doesn't have a name but 20,000 likes, that data is a "signal" we can't ignore just because the database is incomplete.

- **Accuracy**: By deleting that record, we would be lying about the true popularity distribution.

- **Big Picture**: For a journalist, knowing that a very famous but "mysterious" professional exists in the dataset is more valuable information than knowing that a perfectly clean but incomplete dataset exists.

In [72]:
# Using fillna('') to avoid NaN value in return of sum operation
mask = person_details['name'].isna()
person_details.loc[mask, 'name'] = (
    person_details['given_name'].fillna('') + ' ' + person_details['family_name'].fillna('')
).str.strip()

# If it's already empty, we'll put a placeholder
person_details.loc[person_details['name'] == '', 'name'] = "Unknown Professional"

Results:
1. **Efficiency**: We did not lose any data useful for statistical analysis.

2. **Consistency**: We standardized missing names.

3. **Truthful Representation**: The dataset now represents the reality of the original database without artificial distortions.

In [73]:
# 2. NaN management for clarity - substitution for further visualizations
person_details.loc[:, 'relevant_location'] = person_details['relevant_location'].fillna('Unknown')

# 3. Memory optimizations
# View creation without website_rul because of a lot of rows have NaN value in it
#person_details_clean = person_details[['person_mal_id', 'name', 'birthday', 'favorites', 'relevant_location']].copy()

#print(f"Dataset finale pronto per l'analisi: {len(person_details_clean)} righe.")

In [74]:
# Overwriting/importing clean dataframe into .csv
person_details.to_csv('../data/person_details_clean.csv', index=False)
print(f"\nSaved clean person_details dataset")


Saved clean person_details dataset


In [75]:
person_details = None
print(f"\nRemoving person_details dataset from RAM")


Removing person_details dataset from RAM


Implementation ideas:
- **Data integrity visualization**
- **Geographic distribuition - Top 10 locations**
- **Top 10 voice over actors and its folium geographic map**

### Person alternate names

In [76]:
person_alternate_names = pd.read_csv(r'../data/person_alternate_names.csv')
person_alternate_names

,person_mal_id,alt_name
0,1,Seki Mondoya
1,1,門戸 開
2,1,Monto Hiraku
3,3,雪野五月
4,10,Kevin Hatcher
...,...,...
20460,89567,Sydsnap
20461,89567,Queen of Degeneracy
20462,89826,陳浩
20463,89842,Chidori


In [77]:
person_alternate_names.shape

(20465, 2)

In [78]:
person_alternate_names.columns

Index(['person_mal_id', 'alt_name'], dtype='object')

In [79]:
person_alternate_names.dtypes

person_mal_id     int64
alt_name         object
dtype: object

From the preview, we note crucial details regarding the data structure:

- **One-to-Many Relationship**: The same person_mal_id can have multiple alternative names.

- **Data Diversity**: alt_names include both Japanese transcriptions (Kanji) and Western aliases or names.

In [80]:
# 1. Memory Usage
# ID conversion into int32, same as what we did in person_details
person_alternate_names['person_mal_id'] = person_alternate_names['person_mal_id'].astype('int32')

# 2. Checking duplicates (Full duplication)
full_duplicates = person_alternate_names.duplicated().sum()
print(f"Duplicati totali nel dataset alt_names: {full_duplicates}")

# Removing duplicates
person_alternate_names = person_alternate_names.loc[~person_alternate_names.duplicated()]

# 3. Verifying NaNs
print("NaN in alt_names:\n", person_alternate_names.isna().sum())

Duplicati totali nel dataset alt_names: 39
NaN in alt_names:
 person_mal_id     0
alt_name         18
dtype: int64


Here we found some important signals for "Preparation and Investigation" phase.

In [81]:
# Removing record without alternative name
previous_count = len(person_alternate_names)
person_alternate_names = person_alternate_names.dropna(subset=['alt_name'])
current_count = previous_count - len(person_alternate_names)
print(f"Dataset alt_names cleaned from {current_count} rows")

Dataset alt_names cleaned from 18 rows


**Identity Retrieval** (Cross-referencing): the goal is to replace the "Unknown" names in person_details with the first alternative name available in person_alternate_names.

In [82]:
# STEP 1: Create a search dictionary (Mapping)
# Take only the first occurrence of each ID to create a unique name suggestion.
lookup_map = person_alternate_names.drop_duplicates('person_mal_id').set_index('person_mal_id')['alt_name']

# STEP 2: Identifying rows in person_details
person_details = pd.read_csv(r'../data/person_details_clean.csv')
mask = person_details['name'] == "Unknown Professional"

# STEP 3: Detailed recovery
# .map() takes first row ID in person_details to seach the same value in ookup_map
# .fillna() makes us sure about keeping "Unknown Professional" if we don't find an alternative name
person_details.loc[mask, 'name'] = person_details.loc[mask, 'person_mal_id'].map(lookup_map).fillna("Unknown Professional")

# Verifying saved names
recovered_count = (person_details.loc[mask, 'name'] != "Unknown Professional").sum()
print(f"Nomi recuperati con successo: {recovered_count}")

Nomi recuperati con successo: 0


- We found some rows without names and we fill them with 'Unknown Professional', so we thought to cross-referencing datasets to recover missing information; which is a fundamental "Preparation and Investigation" practice. This approach minimizes data loss without introducing bias.
- No artist should remain in the shadows. By cross-analyzing metadata (alternate_names), we were able to restore an identity to professionals who appeared anonymous in the main dataset.
- **Transparency**: It demonstrates accuracy and attention to the audience, who receives more complete and less fragmented data.

In [83]:
# Checking the number of rows that contains"Unknown" in name field
unknown_check = person_details[person_details['name'].str.contains("Unknown", case=False, na=False)]

print(f"Rows that contains 'Unknown': {len(unknown_check)}")
if len(unknown_check) > 0:
    print("Founded name example:")
    print(unknown_check['name'].unique())

Rows that contains 'Unknown': 2
Founded name example:
['Unknown' 'Unknown Professional']


In [84]:
# Taking professional unknown's ID
unknown_ids = unknown_check['person_mal_id'].unique()

# Verifying ID existence in person_alternate_names
matching_ids = person_alternate_names[person_alternate_names['person_mal_id'].isin(unknown_ids)]

print(f"Total amount of unknown ID: {len(unknown_ids)}")
print(f"ID that has an alternative name: {len(matching_ids['person_mal_id'].unique())}")

Total amount of unknown ID: 2
ID that has an alternative name: 0


**Main conclusion: Mismatch**
- The IDs associated with "Unknowns" are not present in the alternate names' dataset.

- Why did this happen? Even with the help of external tables, with over 130 million data points, it's normal for there to be information gaps that cannot be filled.

- **Technical Result**: Since the IDs don't exist in person_alternate_names, the map or merge operation found nothing to "hook," correctly returning 0 retrievals.

In [85]:
# Removal of two irrecoverable outliers for final cleaning -> Clarity
person_details = person_details[~person_details['name'].str.contains("Unknown", case=False, na=False)]

print(f"Dataset finale 'person_details' pulito e verificato: {len(person_details)} righe.")

Dataset finale 'person_details' pulito e verificato: 76592 righe.


In [86]:
# Overwriting/importing clean dataframe into .csv
person_alternate_names.to_csv('../data/person_alternate_names_clean.csv', index=False)
print(f"\nSaved clean person_alternate_names dataset")


Saved clean person_alternate_names dataset


In [87]:
person_details = None
person_alternate_names = None
print(f"\nRemoving person_alternate_names dataset from RAM")


Removing person_alternate_names dataset from RAM


### Person anime works

In [88]:
person_anime_works = pd.read_csv(r'../data/person_anime_works.csv')
person_anime_works

,person_mal_id,position,anime_mal_id
0,1,Theme Song Performance,3080
1,1,Inserted Song Performance,15699
2,1,Theme Song Performance (OP),247
3,1,Theme Song Performance,258
4,1,Theme Song Performance (ED),34825
...,...,...,...
458086,89951,In-Between Animation,11001
458087,89951,Key Animation,55092
458088,89951,"Key Animation (eps 4, 9)",20053
458089,89951,Key Animation,50553


In [89]:
person_anime_works.shape

(458091, 3)

In [90]:
person_anime_works.columns

Index(['person_mal_id', 'position', 'anime_mal_id'], dtype='object')

In [91]:
person_anime_works.dtypes

person_mal_id     int64
position         object
anime_mal_id      int64
dtype: object

In [92]:
# 1. How many roles exist? (Clarity)
print(f"Number of unique roles: {person_anime_works['position'].nunique()}")

# 2. Which are most common rules? (Big Picture)
print("\nTop 10 roles in the industry:")
print(person_anime_works['position'].value_counts().head(10))

Number of unique roles: 80036

Top 10 roles in the industry:
position
Key Animation           29344
Producer                15151
Planning                 9589
Original Creator         9147
Key Animation (ep 1)     8144
Music                    8042
Director                 7805
In-Between Animation     7456
Art Director             5857
Key Animation (ep 2)     5813
Name: count, dtype: int64


In [93]:
# Memory usage
# 1. Converting ID to int32 (Scaling up)
person_anime_works['person_mal_id'] = person_anime_works['person_mal_id'].astype('int32')
person_anime_works['anime_mal_id'] = person_anime_works['anime_mal_id'].astype('int32')

# 2. Position to category (Efficiency)
person_anime_works['position'] = person_anime_works['position'].astype('category')

In [94]:
# Checking NaN per column (Accuracy)
print("NaN per column:\n", person_anime_works.isna().sum())
# Removing if IDs are missing
person_anime_works = person_anime_works.dropna(subset=['person_mal_id', 'anime_mal_id'])

NaN per column:
 person_mal_id    0
position         0
anime_mal_id     0
dtype: int64


In [95]:
# Duplicates management (Accuracy)
duplicates = person_anime_works.duplicated(subset=['person_mal_id', 'position', 'anime_mal_id']).sum()
print(f"\nDuplicates founded: {duplicates}")

if duplicates > 0:
    person_anime_works = person_anime_works.loc[~person_anime_works.duplicated(subset=['person_mal_id', 'position', 'anime_mal_id'])]


Duplicates founded: 0


In [96]:
# Overwriting/importing clean dataframe into .csv
person_anime_works.to_csv('../data/person_anime_works_clean.csv', index=False)
print(f"\nSaved clean person_anime_works dataset")


Saved clean person_anime_works dataset


In [97]:
person_anime_works = None
print(f"\nRemoving person_anime_works dataset from RAM")


Removing person_anime_works dataset from RAM


Implementation and storytelling ideas to understand anime production structure:
1. **The Industry Pillars: Mapping Professional Longevity**: analyzing anime distribution per professional and multi-role
2. **Technical Complexity: Decoding Key Animator Density**: which productions required more workers
3. **Trends during time**: merging data with details.csv to see distribution

### Characters nicknames

In [8]:
character_nicknames = pd.read_csv(r'../data/character_nicknames.csv')
character_nicknames

,character_mal_id,nickname
0,280205,Hikaruko
1,280129,Hinacchi
2,280127,Bertha Willis
3,280066,Jimmy
4,280059,Full Body Red Square
...,...,...
37075,282159,Ling Long
37076,282159,Silvermoon
37077,282227,Mei's Mother
37078,282254,Cyrano


In [4]:
character_nicknames.shape

(37080, 2)

In [5]:
character_nicknames.columns

Index(['character_mal_id', 'nickname'], dtype='object')

In [6]:
character_nicknames.dtypes

character_mal_id     int64
nickname            object
dtype: object

This data represents the aliases or colloquial names by which fans (or different translations) call a character.

Scaling up strategy to save RAM
- **character_mal_id**: This must be converted to an int32 to align with Characters_clean.csv and allow for quick merging.

- **nickname**: This is a string (object). It's not a good idea to use category here because almost every row is a unique name, so we wouldn't save memory.

In [7]:
# 1. ID conversion to int32 (Memory Optimization)
character_nicknames['character_mal_id'] = character_nicknames['character_mal_id'].astype('int32')

# 2. Checking NaN values (Accuracy check)
print("Null values for each column:\n", character_nicknames.isna().sum())

Null values for each column:
 character_mal_id     0
nickname            16
dtype: int64


In [8]:
# 1. Strings cleaning (trim whitespace)
character_nicknames['nickname'] = character_nicknames['nickname'].str.strip()

# 2. Rows removal if nickname is null
character_nicknames = character_nicknames.dropna(subset=['nickname'])

# 3. Duplicates removal (same ID, same nickname)
duplicates = character_nicknames.duplicated(subset=['character_mal_id', 'nickname']).sum()
print(f"Removed duplicated nicknames: {duplicates}")

if duplicates > 0:
    character_nicknames = character_nicknames.drop_duplicates(subset=['character_mal_id', 'nickname'])

Removed duplicated nicknames: 102


In [22]:
# Overwriting/importing clean dataframe into .csv
character_nicknames.to_csv('../data/character_nicknames_clean.csv', index=False)
print(f"\nSaved clean character_nicknames dataset")


Saved clean character_nicknames dataset


In [23]:
character_nicknames = None
print(f"\nRemoving character_nicknames dataset from RAM")


Removing character_nicknames dataset from RAM


Implementing ideas:
1. who is the most loved character? We should seach for number of nicknames for each character
2. "The Multiple Identity": we can create a bar graph that displays the names of real characters instead of their IDs.

### Person voice works

In [1]:
person_voice_works = pd.read_csv(r'../data/person_voice_works.csv')
person_voice_works

NameError: name 'pd' is not defined

In [ ]:
person_voice_works.shape

(489516, 5)

In [ ]:
person_voice_works.columns

Index(['person_mal_id', 'role', 'anime_mal_id', 'character_mal_id',
       'language'],
      dtype='object')

In [ ]:
person_voice_works.dtypes

person_mal_id        int64
role                object
anime_mal_id         int64
character_mal_id     int64
language            object
dtype: object

It's the bridge between characters, anime and dubbing actors, useful to know who makes stories alive and to get the register about every single voice performance.

In [ ]:
# 1. Language distribution (Clarity)
print("Dubbing language distribution:")
print(person_voice_works['language'].value_counts())

Dubbing language distribution:
language
Japanese           203537
English             94902
French              43340
Spanish             39153
Portuguese (BR)     37100
Italian             30712
German              26805
Korean               7981
Mandarin             3529
Hungarian            1378
Hebrew               1079
Name: count, dtype: int64


In [ ]:
# 2. Role distribution (Main vs Supporting)
print("Role distribution:")
print(person_voice_works['role'].value_counts())

Role distribution:
role
Supporting    390055
Main           99461
Name: count, dtype: int64


In [ ]:
# 3. Integrity checking (NaN)
print("Missing values:")
print(person_voice_works.isna().sum())

Missing values:
person_mal_id       0
role                0
anime_mal_id        0
character_mal_id    0
language            0
dtype: int64


- **Original Dominance vs. Global Market**: Japanese accounts for approximately 41.5% of performances, but the remaining 58.5% is distributed in international markets, with English clearly leading the dubbing front.

- **Cast Structure**: The ratio of supporting roles to protagonists is approximately 4:1. This means that for every hero ("Main"), there are on average four supporting characters who enrich the narrative world.

- **Total Integrity**: The absence of null values allows us to perform complex merges without the risk of losing records along the way, ensuring the scalability of future analyses.

In [57]:
# 1. Memory optimization (Scaling Strategy)
# Converting ID into int32
id_cols = ['person_mal_id', 'anime_mal_id', 'character_mal_id']
for col in id_cols:
    person_voice_works[col] = person_voice_works[col].astype('int32')

# Category variables (Role e Language due to their repetition)
person_voice_works['role'] = person_voice_works['role'].astype('category')
person_voice_works['language'] = person_voice_works['language'].astype('category')

# 2. Duplicates managing
# None can dub same character in the same anime with same language twice
duplicates = person_voice_works.duplicated().sum()
print(f"Total duplicates removed: {duplicates}")
person_voice_works = person_voice_works.drop_duplicates()

# 3. NaN removal
person_voice_works = person_voice_works.dropna(subset=['person_mal_id', 'character_mal_id'])

NameError: name 'person_voice_works' is not defined

In [ ]:
# Overwriting/importing clean dataframe into .csv
person_voice_works.to_csv('../data/person_voice_works_clean.csv', index=False)
print(f"\nSaved clean person_voice_works dataset")


Saved clean person_voice_works dataset


In [ ]:
person_voice_works = None
print(f"\nRemoving person_voice_works dataset from RAM")


Removing person_voice_works dataset from RAM


Implementing ideas:
- **Anime's Tower of Babel**: A chart showing the prevalence of languages. It'd be interesting to see how much non-Japanese dubs (English, French, German, etc.) are represented.
- **Polyglot dubbing actors**
- **Main vs. Supporting**: A chart comparing a voice actor's career: how often are they in the spotlight (Main) versus supporting roles?
- **Cross-Over Identity**: By cross-referencing this file with character_nicknames_clean.csv, we might find out if characters with multiple nicknames tend to have multiple different voice actors over time.
- **Top 10 dubbing actors details'**

### Stats

Visualization and first considerations of the dataset structure

In [ ]:
stats = pd.read_csv(r'../data/stats.csv')
stats

In [ ]:
stats.shape

In [ ]:
stats.columns

In [ ]:
stats.dtypes

### Missing Data Assessment

The following code calculates both the absolute count and the relative percentage of missing values (`NaN`) across all columns. It allows us to make informed, data-driven decisions on how to properly handle these gaps (e.g., through specific imputation techniques or by dropping uninformative records) to ensure the reliability of our subsequent analysis and visualizations.

In [ ]:
# Check for missing values in each column
print("Missing values:")
print(stats.isnull().sum())

# Check the percentage of missing data
print("\nPercentage of missing values:")
print((stats.isnull().sum() / len(stats)) * 100)

Check the mal_id key for duplicate anime

In [ ]:
# Check for duplicate mal_id
duplicates_count = stats.duplicated(subset=['mal_id']).sum()
print(f"Number of duplicate mal_id: {duplicates_count}")

### Inspecting Records with Missing Data

To better understand the nature of our missing values, we isolate the specific rows within the dataset that contain at least one `NaN` (using `axis=1`). Rather than displaying the entire dataframe, we filter our view to a targeted subset of user engagement columns (such as `watching`, `completed`, and `dropped`) along with the anime's unique identifier (`mal_id`). 
By sampling the first 20 occurrences, we can visually investigate potential patterns in the missingness. This qualitative inspection helps us determine the statistics andament.

We can see that the only column with outliers is `plan_to_watch`, which suggests that these users with NaN values ​​in the rating columns are either planning to watch anime or planning to watch anime that has yet to be released.
Let's leave this type of user alone so we can investigate their behavior with the total user count.

In [ ]:
# list of desired columns
interest_columns =[ 'mal_id', 'watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch', 'total']

# Display rows that have NaN values
visualization = stats[stats.isna().any(axis=1)][interest_columns]

print(visualization.head(20))


Searching if there are some rating with negative values

In [81]:
negative_values = stats[(stats[stats_columns] < 0).any(axis=1)]

if not negative_values.empty:
    print(f"Found {len(negative_values)} rows with negative counts")
else:
    print("No negative values found")

No negative values found


### Memory Optimization and Data Type Casting

In preparation for merging our datasets, we perform a crucial memory optimization step. The user interaction metrics (such as `watching`, `completed`, and `dropped`) inherently represent discrete counts of people. For this, we downcast them to `int32`. 

For the rating distributions (`score_1_votes` through `score_10_votes`), we utilize Pandas' nullable integer data type (`Int32`). This strategic choice allows us to significantly reduce the memory footprint while gracefully preserving any missing values (`<NA>`) that might still exist in the voting data. Finally, we output the `dtypes` to verify that the casting was successfully applied across all selected columns.

In [ ]:
# Convert counts to int32 as they are discrete values (people)
# This optimizes memory usage before merging with other dataframes
columns_to_optimize = ['mal_id', 'watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch', 'total']

for col in columns_to_optimize:
    stats[col] = stats[col].astype('int32')

colonne_score = [f'score_{i}_votes' for i in range(1, 11)]
stats[colonne_score] = stats[colonne_score].astype('Int32')
stats.dtypes

In [83]:
# Overwriting/importing clean dataframe into .csv
stats.to_csv('../data/stats_clean.csv', index=False)
print(f"\nSaved clean stats dataset")


Saved clean stats dataset


In [84]:
stats = None
print(f"\nRemoving stats dataset from RAM")


Removing stats dataset from RAM
